In [1]:
def load_pages(path="pages.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["pages"]


In [2]:
def fetch_wikipedia_summary(title):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
    headers = {"User-Agent": "chroma-wiki-hybrid/1.0"}

    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        return None

    return r.json()


In [3]:
def chunk_text(text, max_words=120):
    words = text.split()
    return [
        " ".join(words[i:i + max_words])
        for i in range(0, len(words), max_words)
    ]


In [4]:
from sentence_transformers import SentenceTransformer

class SentenceTransformerEmbedding:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self._model_name = model_name

    def __call__(self, input):
        return self.model.encode(input).tolist()

    def embed_query(self, input):
        return self.model.encode(input).tolist()

    def name(self):
        return f"sentence-transformers-{self._model_name}"

    def get_config(self):
        return {"model_name": self._model_name}


In [5]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(
    Settings(
        persist_directory="./wiki_chroma",
        anonymized_telemetry=False
    )
)

embedding_function = SentenceTransformerEmbedding()

collection = client.get_or_create_collection(
    name="wiki_fixed_pages",
    embedding_function=embedding_function
)


In [6]:
def load_queries(path="queries.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["queries"]


In [7]:
# def ingest_pages(pages, collection):
#     docs, metadatas, ids = [], [], []

#     for title in pages:
#         data = fetch_wikipedia_summary(title)
#         if not data or "extract" not in data:
#             continue

#         chunks = chunk_text(data["extract"])

#         for i, chunk in enumerate(chunks):
#             docs.append(chunk)
#             metadatas.append({
#                 "title": title,
#                 "page_id": data.get("pageid"),
#                 "source": "wikipedia"
#             })
#             ids.append(f"{data.get('pageid')}_{i}")

#     collection.add(
#         documents=docs,
#         metadatas=metadatas,
#         ids=ids
#     )


In [8]:
# def run_queries(collection, queries, k=3):
#     for q in queries:
#         res = collection.query(
#             query_texts=[q],
#             n_results=k
#         )

#         print(f"\nQUERY: {q}")
#         for doc in res["documents"][0]:
#             print(" -", doc[:120], "...")


In [9]:
# import json, requests

# pages = load_pages("pages.json")
# ingest_pages(pages, collection)

# print("Total chunks:", collection.count())

# queries = load_queries("queries.json")
# run_queries(collection, queries)a


In [10]:
import time
from collections import defaultdict


In [17]:
def ingest_pages_verbose(pages, collection):
    print("=== INGESTION STARTED ===")
    start_time = time.perf_counter()

    total_chunks = 0

    for idx, title in enumerate(pages, 1):
        print(f"[INGEST] ({idx}/{len(pages)}) Fetching page: {title}")

        page_start = time.perf_counter()
        data = fetch_wikipedia_summary(title)

        if not data or "extract" not in data:
            print(f"  ⚠️ Skipped (no extract)")
            continue

        chunks = chunk_text(data["extract"])
        page_id = data.get("pageid")

        docs, metadatas, ids = [], [], []

        for i, chunk in enumerate(chunks):
            docs.append(chunk)
            metadatas.append({
                "title": title,
                "page_id": page_id,
                "source": "wikipedia"
            })
            ids.append(f"{page_id}_{i}")

        if docs:
            collection.add(
                documents=docs,
                metadatas=metadatas,
                ids=ids
            )
            print(f"Added {len(docs)} chunks")
        else:
            print(" No valid chunks - skipped add()")

        page_time = time.perf_counter() - page_start
        total_chunks += len(docs)

        print(
            f" Added {len(docs)} chunks "
            f"(page latency: {page_time:.2f}s)"
        )

    total_time = time.perf_counter() - start_time
    throughput = total_chunks / total_time if total_time > 0 else 0

    print("\n=== INGESTION COMPLETED ===")
    print(f"Total pages ingested : {len(pages)}")
    print(f"Total chunks         : {total_chunks}")
    print(f"Total time (s)       : {total_time:.2f}")
    print(f"Throughput (chunks/s): {throughput:.2f}")

    return {
        "pages": len(pages),
        "chunks": total_chunks,
        "time": total_time,
        "throughput": throughput
    }


In [18]:
def run_queries_verbose(collection, queries, k=3):
    print("\n=== QUERYING STARTED ===")

    query_latencies = []
    per_query_results = defaultdict(list)

    start_time = time.perf_counter()

    for i, q in enumerate(queries, 1):
        print(f"\n[QUERY] ({i}/{len(queries)}) {q}")

        q_start = time.perf_counter()
        res = collection.query(
            query_texts=[q],
            n_results=k,
            include=["documents", "metadatas"]
        )
        q_time = time.perf_counter() - q_start
        query_latencies.append(q_time)

        docs = res["documents"][0]
        metas = res["metadatas"][0]

        for rank, (doc, meta) in enumerate(zip(docs, metas), 1):
            print(f"  → Rank {rank}")
            print(f"     Page : {meta['title']}")
            print(f"     Text : {doc[:120]}...")

            per_query_results[q].append({
                "rank": rank,
                "page": meta["title"],
                "text": doc
            })

        print(f"Query latency: {q_time:.3f}s")

    total_time = time.perf_counter() - start_time
    throughput = len(queries) / total_time if total_time > 0 else 0

    print("\n=== QUERYING COMPLETED ===")
    print(f"Total queries        : {len(queries)}")
    print(f"Total time (s)       : {total_time:.2f}")
    print(f"Avg latency (s)      : {sum(query_latencies)/len(query_latencies):.3f}")
    print(f"Throughput (q/s)     : {throughput:.2f}")

    return {
        "queries": len(queries),
        "avg_latency": sum(query_latencies) / len(query_latencies),
        "throughput": throughput,
        "results": per_query_results
    }


In [13]:
def reset_collection(client, collection_name):
    try:
        client.delete_collection(collection_name)
        print(f"Collection '{collection_name}' deleted.")
    except Exception as e:
        print(f"Collection '{collection_name}' did not exist or already deleted.")


In [14]:
# reset_collection(client, "wiki_fixed_pages")


In [16]:
import json, requests

pages = load_pages("pages.json")
queries = load_queries("queries.json")

# Ingest with live progress + metrics
ingest_stats = ingest_pages_verbose(pages, collection)

print("\nCurrent collection size:", collection.count())

# Query with attribution + metrics
query_stats = run_queries_verbose(collection, queries)


=== INGESTION STARTED ===
[INGEST] (1/99) Fetching page: AI safety
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.39s)
[INGEST] (2/99) Fetching page: Peripheral
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.26s)
[INGEST] (3/99) Fetching page: Naïve physics
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.24s)
[INGEST] (4/99) Fetching page: Anomaly detection
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.24s)
[INGEST] (5/99) Fetching page: 80 Million Tiny Images
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.25s)
[INGEST] (6/99) Fetching page: AI Overviews
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.25s)
[INGEST] (7/99) Fetching page: Joaquim da Costa Ribeiro
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.27s)
[INGEST] (8/99) Fetching page: Biology
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.23s)
[INGEST] (9/99) Fetching page: Physics of Life
Added 1 chunks
  ✔ Added 1 chunks (page latency: 0.22s)
[INGEST] (10/99) Fetching page: Category:Computer sci